In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from prophet import Prophet


In [ ]:
df = pd.read_csv("dataset_preparasi.csv")

# convert date
df["Tanggal PO"] = pd.to_datetime(df["Tanggal PO"], errors="coerce")

# revenue
df["revenue"] = df["nilai_po"]

# drop null date
df = df.dropna(subset=["Tanggal PO"])

# basic check
df.info()
df.head()


#Descriptive Analysis

##Total Revenue

In [ ]:
df["revenue"].sum()


##Revenue by Year

In [ ]:
rev_year = df.groupby("Tahun")["revenue"].sum()

rev_year.plot(kind="bar", colormap='viridis', title="Revenue per Year")
plt.show()

##Top Customers

In [ ]:
top_customers = df.groupby("Nama Customer")["revenue"].sum().sort_values(ascending=False).head(10)
top_customers


##Top Product

In [ ]:
top_products = df.groupby("Nama Barang")["revenue"].sum().sort_values(ascending=False).head(10)
top_products


#Customer Feature engineering (RFM)

In [ ]:
snapshot_date = df["Tanggal PO"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Nama Customer").agg({
    "Tanggal PO": lambda x: (snapshot_date - x.max()).days,
    "No.PO": "nunique",
    "revenue": ["sum","mean"],
    "Nama Barang": "nunique"
})

rfm.columns = ["recency","frequency","monetary","avg_order","product_diversity"]
rfm.reset_index(inplace=True)

rfm.head()


#CLTV Estimation (Regression)

In [ ]:
X = rfm[["recency","frequency","monetary","product_diversity"]]
y = rfm["monetary"] * 1.3  # proxy future value

from xgboost import XGBRegressor

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

cltv_model = XGBRegressor()
cltv_model.fit(X_train,y_train)

rfm["CLTV_pred"] = cltv_model.predict(X)


#Retention Model

In [ ]:
rfm["active_next_year"] = np.where(rfm["recency"]<365,1,0)

X_ret = rfm[["recency","frequency","monetary","product_diversity"]]
y_ret = rfm["active_next_year"]

ret_model = LogisticRegression()
ret_model.fit(X_ret,y_ret)

rfm["retention_prob"] = ret_model.predict_proba(X_ret)[:,1]



#Customer Segmentation (Kmean)

In [ ]:
seg_features = rfm[["CLTV_pred","retention_prob","frequency","product_diversity"]]

scaled = StandardScaler().fit_transform(seg_features)

kmeans = KMeans(n_clusters=4, random_state=42)
rfm["segment"] = kmeans.fit_predict(scaled)

segment_map = {
    0:"Loyal",
    1:"Growth",
    2:"Risk",
    3:"Dormant"
}

rfm["segment_name"] = rfm["segment"].map(segment_map)

rfm["segment_name"].value_counts()


#Product Demand Forecast (Prophet)

In [ ]:
prod_ts = df.groupby(["Tanggal PO","Nama Barang"])["revenue"].sum().reset_index()

forecast_list = []

for product in prod_ts["Nama Barang"].unique():
    temp = prod_ts[prod_ts["Nama Barang"]==product]
    temp = temp.rename(columns={"Tanggal PO":"ds","revenue":"y"})

    if len(temp) < 10:
        continue

    m = Prophet()
    m.fit(temp)

    future = m.make_future_dataframe(periods=12,freq="M")
    forecast = m.predict(future)

    forecast_list.append({
        "Nama Barang": product,
        "forecast_revenue": forecast["yhat"].tail(12).sum()
    })

forecast_df = pd.DataFrame(forecast_list)
forecast_df.head()


#OPPORTUNITY SCORE ENGINE

In [ ]:
rfm["opportunity_score"] = rfm["CLTV_pred"] * rfm["retention_prob"]

forecast_df["product_score"] = forecast_df["forecast_revenue"] / forecast_df["forecast_revenue"].max()


#Customer x Product Matrix

In [ ]:
cust_prod = df.groupby(["Nama Customer","Nama Barang"])["revenue"].sum().reset_index()

cust_prod = cust_prod.merge(
    rfm[["Nama Customer","opportunity_score","segment_name"]],
    on="Nama Customer"
)

cust_prod = cust_prod.merge(
    forecast_df[["Nama Barang","product_score"]],
    on="Nama Barang",
    how="left"
)

cust_prod["future_value_est"] = (
    cust_prod["revenue"] *
    cust_prod["opportunity_score"] *
    cust_prod["product_score"].fillna(0)
)

cust_prod.sort_values("future_value_est",ascending=False).head(20)


#Final Output

In [ ]:
rfm.sort_values("CLTV_pred",ascending=False)[["Nama Customer","CLTV_pred","segment_name"]].head(10)


##Top Product Masa Depan

In [ ]:
forecast_df.sort_values("forecast_revenue",ascending=False).head(10)


In [ ]:
cust_prod.sort_values("future_value_est",ascending=False).head(20)


#Visual Segmentasi Customer (Kmeans)

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=rfm,
    x="frequency",
    y="CLTV_pred",
    hue="segment_name",
    palette="Set2"
)
plt.title("Customer Segmentation: Frequency vs CLTV")
plt.xlabel("Order Frequency")
plt.ylabel("Predicted CLTV")
plt.show()


# Kiri bawah = low value
# Kanan atas = customer inti
# Warna = loyal / growth / risk / dormant


#CLTV vs RETENTION PROBABILITY

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=rfm,
    x="retention_prob",
    y="CLTV_pred",
    hue="segment_name",
    palette="Set1"
)
plt.title("Customer Value vs Retention Probability")
plt.xlabel("Retention Probability")
plt.ylabel("Predicted CLTV")
plt.show()

# Kanan atas = PRIORITAS SALES
# Kiri atas = mahal tapi rawan hilang
# Kanan bawah = murah tapi stabil


#FORECAST DEMAND (CONTOH 1 PRODUK)

In [ ]:
sample_product = forecast_df.sort_values("forecast_revenue",ascending=False).iloc[0]["Nama Barang"]

temp = prod_ts[prod_ts["Nama Barang"]==sample_product]
temp = temp.rename(columns={"Tanggal PO":"ds","revenue":"y"})

m = Prophet()
m.fit(temp)

future = m.make_future_dataframe(periods=12,freq="M")
forecast = m.predict(future)

m.plot(forecast)
plt.title(f"Demand Forecast: {sample_product}")
plt.show()


#OPPORTUNITY MATRIX (CUSTOMER × PRODUCT)

In [ ]:
top_opportunity = cust_prod.sort_values("future_value_est",ascending=False).head(20)

plt.figure(figsize=(10,6))
sns.barplot(
    data=top_opportunity,
    y="Nama Customer",
    x="future_value_est",
    hue="Nama Barang"
)
plt.title("Top Revenue Opportunities (Customer x Product)")
plt.xlabel("Estimated Future Revenue")
plt.ylabel("Customer")
plt.show()


#Segment Distribution

In [ ]:
rfm["segment_name"].value_counts().plot.pie(autopct="%1.1f%%", figsize=(6,6))
plt.title("Customer Segment Distribution")
plt.ylabel("")
plt.show()


In [ ]:
df["product_type"] = df["Nama Barang"].apply(
    lambda x: "instrument" if "analyzer" in x.lower() or "meter" in x.lower() else "consumable"
)

rev_type = df.groupby("product_type")["revenue"].sum()

rev_type.plot(kind="bar",title="Revenue by Product Type")
plt.ylabel("Revenue")
plt.show()


#Evaluasi Prophet

In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# agregasi ke level bulanan per produk
ts = df.groupby(["Bulan", "Nama Barang"])["nilai_po"].sum().reset_index()

# pilih 1 produk contoh
product_name = ts["Nama Barang"].value_counts().idxmax()

temp = ts[ts["Nama Barang"] == product_name].copy()
temp["ds"] = pd.to_datetime(temp["Bulan"].astype(str))
temp = temp.rename(columns={"nilai_po": "y"})

# sort time
temp = temp.sort_values("ds").reset_index(drop=True)


In [ ]:
n_test = 6

train = temp.iloc[:-n_test]
test  = temp.iloc[-n_test:]


In [ ]:
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

m.fit(train)


In [ ]:
future = m.make_future_dataframe(periods=n_test, freq="M")
forecast = m.predict(future)

forecast_test = forecast.tail(n_test)[["ds", "yhat"]].reset_index(drop=True)
test = test.reset_index(drop=True)


In [ ]:
mape = mean_absolute_percentage_error(test["y"], forecast_test["yhat"])
rmse = np.sqrt(mean_squared_error(test["y"], forecast_test["yhat"]))

print(f"MAPE  : {mape:.2%}")
print(f"RMSE  : {rmse:,.0f}")


###MAPE < 20% → bagus
###20–40% → moderat
###40% → lemah (perlu agregasi kategori/brand)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train["ds"], train["y"], label="Train")
plt.plot(test["ds"], test["y"], label="Actual (Test)")
plt.plot(forecast_test["ds"], forecast_test["yhat"], label="Forecast", linestyle="--")
plt.legend()
plt.title(f"Prophet Forecast Evaluation: {product_name}")
plt.xlabel("Time")
plt.ylabel("Revenue")
plt.show()


In [ ]:
results = []

for product in ts["Nama Barang"].unique():
    temp = ts[ts["Nama Barang"] == product].copy()
    temp["ds"] = pd.to_datetime(temp["Bulan"].astype(str))
    temp = temp.rename(columns={"nilai_po": "y"})
    temp = temp.sort_values("ds")

    if len(temp) < 12:
        continue  # data terlalu pendek

    train = temp.iloc[:-6]
    test = temp.iloc[-6:]

    m = Prophet()
    m.fit(train)

    future = m.make_future_dataframe(periods=6, freq="M")
    forecast = m.predict(future)

    forecast_test = forecast.tail(6)["yhat"]

    mape = mean_absolute_percentage_error(test["y"], forecast_test)

    results.append({
        "Nama Barang": product,
        "MAPE": mape
    })

eval_df = pd.DataFrame(results).sort_values("MAPE")
eval_df.head(10)


In [ ]:
df["product_type"].value_counts()

In [ ]:
ts = df.groupby(["Bulan", "product_type"])["nilai_po"].sum().reset_index()

ts["ds"] = pd.to_datetime(ts["Bulan"].astype(str))
ts = ts.rename(columns={"nilai_po": "y"})

ts = ts.sort_values("ds")
ts.head()


In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

results = []

for ptype in ts["product_type"].unique():
    temp = ts[ts["product_type"] == ptype].copy()
    temp = temp.sort_values("ds")

    # minimal data check
    if len(temp) < 12:
        continue

    # time-based split (6 bulan terakhir = test)
    n_test = 6
    train = temp.iloc[:-n_test]
    test  = temp.iloc[-n_test:]

    m = Prophet(yearly_seasonality=True)
    m.fit(train)

    future = m.make_future_dataframe(periods=n_test, freq="M")
    forecast = m.predict(future)

    forecast_test = forecast.tail(n_test)[["ds", "yhat"]].reset_index(drop=True)
    test = test.reset_index(drop=True)

    mape = mean_absolute_percentage_error(test["y"], forecast_test["yhat"])
    rmse = np.sqrt(mean_squared_error(test["y"], forecast_test["yhat"]))

    results.append({
        "product_type": ptype,
        "MAPE": mape,
        "RMSE": rmse
    })

    # ===== VISUAL =====
    plt.figure(figsize=(8,4))
    plt.plot(train["ds"], train["y"], label="Train")
    plt.plot(test["ds"], test["y"], label="Actual")
    plt.plot(forecast_test["ds"], forecast_test["yhat"], label="Forecast", linestyle="--")
    plt.title(f"Forecast Evaluation: {ptype}")
    plt.legend()
    plt.show()

results_df = pd.DataFrame(results)
results_df


In [ ]:
ptype = "consumable"

temp = ts[ts["product_type"] == ptype].copy()

m = Prophet(yearly_seasonality=True)
m.fit(temp)

future = m.make_future_dataframe(periods=12, freq="M")
forecast = m.predict(future)

m.plot(forecast)
plt.title(f"Forecast Future Demand: {ptype}")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from prophet import Prophet
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score



In [ ]:
df["Tanggal PO"] = pd.to_datetime(df["Tanggal PO"], errors="coerce")
df["Tahun"] = df["Tanggal PO"].dt.year
df["Bulan"] = df["Tanggal PO"].dt.to_period("M")

df["Kategori Product"] = df["Kategori Product"].str.lower().str.strip()
df["Nama Customer"] = df["Nama Customer"].str.strip()
df["Market"] = df["Market"].fillna("Unknown")
df["Brand"] = df["Brand"].fillna("Unknown")

df = df[df["nilai_po"] > 0]

In [ ]:
df.info(
)

In [ ]:
df["product_type"] = np.where(
    df["Kategori Product"].str.contains("instrument|analyzer|meter", regex=True),
    "instrument",
    "consumable"
)


In [ ]:
snapshot_date = df["Tanggal PO"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Nama Customer").agg({
    "Tanggal PO": lambda x: (snapshot_date - x.max()).days,
    "No.PO": "nunique",
    "nilai_po": "sum"
}).reset_index()

rfm.columns = ["Nama Customer","Recency","Frequency","Monetary"]

rfm["AOV"] = rfm["Monetary"] / rfm["Frequency"]
rfm["CLTV"] = rfm["AOV"] * rfm["Frequency"] * 2   # horizon 2 tahun



In [ ]:
ts = (
    df.groupby(["Bulan","product_type"])["nilai_po"]
    .sum()
    .reset_index()
)

ts["ds"] = ts["Bulan"].dt.to_timestamp()
ts["y"] = ts["nilai_po"]

forecast_results = []

for ptype in ts["product_type"].unique():
    sub = ts[ts["product_type"]==ptype][["ds","y"]]
    train = sub.iloc[:-3]

    # Skip if training data is insufficient
    if len(train) < 2:
        continue

    model = Prophet()
    model.fit(train)

    future = model.make_future_dataframe(12, freq="M")
    fc = model.predict(future)

    forecast_sum = fc.tail(12)["yhat"].sum()

    forecast_results.append({
        "product_type": ptype,
        "forecast_revenue": forecast_sum
    })

forecast_df = pd.DataFrame(forecast_results)
forecast_df

In [ ]:
df["is_instrument"] = (df["product_type"]=="instrument").astype(int)

cust_year = df.groupby(["Nama Customer","Tahun"]).agg({
    "is_instrument":"max",
    "nilai_po":"sum",
    "No.PO":"nunique"
}).reset_index()

cust_year["target"] = cust_year.groupby("Nama Customer")["is_instrument"].shift(-1)
cust_year = cust_year.dropna()


In [ ]:
X = cust_year[["nilai_po","No.PO","is_instrument"]]
y = cust_year["target"]

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X,y)

# Check if the model was trained on more than one class
if len(model.classes_) > 1:
    # If there are two classes, get the probability of class 1
    cust_year["prob_buy_instrument"] = model.predict_proba(X)[:,1]
else:
    # If only one class was present in the target variable, all probabilities for class 1 are 0
    # (assuming the single class is 0, which is the case based on the kernel state)
    cust_year["prob_buy_instrument"] = 0.0
    print("Warning: Target variable 'y' contains only one class. 'prob_buy_instrument' set to 0.0 for all entries.")

In [ ]:
cust_prob = cust_year.groupby("Nama Customer")["prob_buy_instrument"].mean().reset_index()

customer_model = rfm.merge(cust_prob, on="Nama Customer", how="left")
customer_model["prob_buy_instrument"] = customer_model["prob_buy_instrument"].fillna(0)


In [ ]:
avg_instrument_value = df[df["product_type"]=="instrument"]["nilai_po"].mean()

customer_model["expected_instrument_rev"] = (
    customer_model["prob_buy_instrument"] * avg_instrument_value
)

customer_model["expected_total_rev"] = (
    customer_model["Monetary"] + customer_model["expected_instrument_rev"]
)


# Task
I will impute the missing values in `features` with the median of each column, then standardize these features using `StandardScaler`, and finally apply `KMeans` clustering to assign segment labels.

## Impute missing values in features

### Subtask:
Fill any NaN values in the `features` DataFrame with the median of their respective columns to prepare for scaling and clustering.


**Reasoning**:
The `features` DataFrame contains NaN values, which caused an error during KMeans clustering. I need to impute these missing values before proceeding. The instruction specifies to fill NaNs with the median of their respective columns.



In [ ]:
median_values = features.median()
features = features.fillna(median_values)

print("NaN values after imputation:")
print(features.isnull().sum())

**Reasoning**:
With the NaN values in the 'features' DataFrame now imputed, the data is ready for standardization. This step will scale the features to have a mean of 0 and a standard deviation of 1, which is crucial for K-Means clustering to perform accurately as it relies on distance metrics.



In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print("Features scaled successfully. Shape:")
print(X_scaled.shape)

**Reasoning**:
Now that the features have been scaled, I will apply KMeans clustering to the `X_scaled` data to segment the customers. The resulting cluster labels will be added as a new column, 'segment', to the `customer_model` DataFrame.



In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
customer_model["segment"] = kmeans.fit_predict(X_scaled)

print("Customer segmentation completed successfully. Segment distribution:")
print(customer_model["segment"].value_counts())

## Summary:

### Data Analysis Key Findings

*   Missing values (NaNs) in the `features` DataFrame were successfully imputed using the median of each respective column, resulting in zero remaining NaN values.
*   The imputed features were standardized using `StandardScaler`, preparing them for clustering.
*   `KMeans` clustering with four clusters was applied to the scaled features, and the resulting segment labels were added to the `customer_model` DataFrame.
*   The distribution of customers across the four segments is as follows: Segment 0 contains 166 customers, Segment 1 contains 135 customers, Segment 3 contains 12 customers, and Segment 2 contains 2 customers.

### Insights or Next Steps

*   The customer segmentation has been successfully completed, providing a foundation for understanding distinct customer groups for targeted strategies.
*   Given the highly uneven distribution of customers across the segments (e.g., Segment 2 has only 2 customers while Segment 0 has 166), it would be beneficial to perform further analysis to understand the characteristics of each segment and re-evaluate the optimal number of clusters if these small segments are not analytically useful.


In [ ]:
segment_map = {
    0: "Strategic Core",
    1: "Growth Target",
    2: "Maintenance",
    3: "Churn Risk"
}

customer_model["segment_name"] = customer_model["segment"].map(segment_map)


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=customer_model,
    x="Monetary",
    y="prob_buy_instrument",
    hue="segment_name"
)
plt.title("Customer Segmentation")
plt.show()


In [ ]:
customer_model.sort_values("expected_total_rev", ascending=False).head(15)


In [ ]:
brand_seg = df.merge(customer_model[["Nama Customer","segment_name"]], on="Nama Customer")

brand_perf = brand_seg.groupby(["Brand","segment_name"])["nilai_po"].sum().reset_index()

brand_perf.sort_values("nilai_po", ascending=False).head(10)
